# Error Handling & Reading Tracebacks

When you start calling LLMs and APIs, things **will** go wrong: a wrong API key, an empty response, no internet, a user typing garbage. A beginner sees a wall of red text and panics. A developer reads that red text like a story and fixes it in 10 seconds.

This lesson teaches you to:
1. **Read** an error (the traceback) instead of fearing it
2. **Handle** errors so your app doesn't crash (`try` / `except`)

> This is also the #1 skill for *vibe coding*: when the AI's code breaks, you paste the error back and fix it. So you must be able to read errors.

## 1. What happens when code crashes

Remember our margin formula from lesson 1: `margin = profit * 100 / revenue`. What if revenue is 0? Let's see what Python does.

In [ ]:
revenue = 0
expenses = 10

profit = revenue - expenses
margin = profit * 100 / revenue   # dividing by zero!
print(margin)

Congratulations — you just shipped your first bug. 🐛 That red text isn't Python yelling at you... okay, it's yelling a *little*. But it's actually trying to help. Let's learn to read it instead of running away.

## 2. How to READ a traceback

You'll get a red block like this:

```
---------------------------------------------------------------------------
ZeroDivisionError                         Traceback (most recent call last)
Cell In[1], line 5
      3 profit = revenue - expenses
----> 5 margin = profit * 100 / revenue

ZeroDivisionError: division by zero
```

**Read it BOTTOM to TOP:**
- **Last line** = the most important part: the *error type* (`ZeroDivisionError`) and a plain-English *message* (`division by zero`).
- **The arrow `---->`** points to the exact line that broke (line 5).
- **Lines above** show the path your code took to get there (useful when functions call functions).

Rule of thumb: **read the last line first, then look at the arrow.** That alone solves 80% of errors. If you're stuck, copy the last line into Google or your AI assistant.

### Common error types you'll meet

In [ ]:
int("hello")   # ValueError: can't turn text into a number

In [ ]:
fin = {"revenue": 50, "expenses": 10}
fin["profit"]   # KeyError: that key doesn't exist

In [ ]:
revenues = [50, 60, 55]
revenues[10]   # IndexError: only 3 items, no index 10

| Error type | What it usually means |
|---|---|
| `ZeroDivisionError` | You divided by 0 |
| `ValueError` | Right type, wrong value (e.g. `int("hello")`) |
| `TypeError` | Wrong type (e.g. `"5" + 5`) |
| `KeyError` | Dictionary key doesn't exist |
| `IndexError` | List index out of range |
| `NameError` | Using a variable/typo that isn't defined |
| `ModuleNotFoundError` | You forgot to `pip install` something |

Don't bother memorizing this table. 90% of a programmer's life is *creating* these errors, and the other 10% is googling them. You'll know them all by heart soon enough. 😄

## 3. try / except: don't let the app crash

Instead of crashing, we *try* the risky code and have a backup plan in *except*.

> **Analogy:** `try/except` is like a dosa shop running on a generator. If the power cuts out (error), they switch to the generator (`except`) and keep serving — instead of locking the shutters and sending everyone home (crash).

In [ ]:
revenue = 0
expenses = 10

try:
    margin = (revenue - expenses) * 100 / revenue
    print(f"Margin: {margin:.2f}%")
except ZeroDivisionError:
    print("Revenue is 0, can't calculate margin. Skipping.")

print("Program keeps running 🎉")

### Catch specific errors (recommended) vs catch-all

You can name the exact error, or catch everything with a bare `except Exception`. Catching the *specific* one is better — you only handle problems you actually expect.

In [ ]:
def safe_margin(revenue, expenses):
    try:
        return (revenue - expenses) * 100 / revenue
    except ZeroDivisionError:
        return None   # signal "couldn't compute"

print(safe_margin(50, 10))
print(safe_margin(0, 10))

In [ ]:
# Grab the error object with 'as e' to see the message
try:
    age = int("twenty")
except ValueError as e:
    print("Couldn't convert to a number.")
    print("Technical detail:", e)

## 4. Why this matters for AI apps: handling LLM calls

This is the real reason we learn it now. When you call an LLM, lots can go wrong that is **not your fault**:
- wrong / missing API key
- no internet
- the API is down or you hit a rate limit
- the response comes back empty

If any of these happen, your Streamlit recipe app would crash and show the user a scary stack trace. Let's wrap our `generate_recipe` from lesson 3 so it fails *gracefully*.

In [ ]:
from dotenv import load_dotenv
from google import genai

load_dotenv()
client = genai.Client()

def generate_recipe(ingredients, cuisine, diet):
    prompt = f'''
    Generate one food recipe using these ingredients: {ingredients}
    Recipe should not be more than 100 words
    Cuisine: {cuisine}
    Diet: {diet}
    '''
    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )
        if not response.text:
            return "The model returned an empty response. Please try again."
        return response.text
    except Exception as e:
        # Network down, bad API key, rate limit, etc.
        return f"Sorry, couldn't generate a recipe right now. ({e})"

print(generate_recipe("tomatoes, lentils, onion", "Indian", "Vegan"))

Trust me on this one: your API key has a magical talent for going 'missing' at the *exact* moment you're demoing to your boss. `try/except` is your insurance policy. 😅

Now your app shows a friendly message instead of crashing. In Streamlit you'd do `st.error(...)` instead of returning the text. **This is the difference between a demo and a real app.**

## Exercise: Codebasics Chai Stall — Resilient Bill Calculator

Remember the chai bill calculator from lesson 2? Customers (and bugs) don't always behave. Make it bulletproof.

Write a function `safe_chai_bill(cups, price_per_cup)` that:
1. Converts `cups` and `price_per_cup` to numbers using `int()` / `float()` (they may arrive as text from a form).
2. Uses `try/except` to catch a `ValueError` if someone passes `"two"` instead of `2` → return the message `"Please enter numbers only."`
3. If `cups` is less than 1, return the message `"Cups must be at least 1"`.
4. Otherwise return the total bill (`cups * price_per_cup`).

Test it with these calls:
```python
print(safe_chai_bill("3", "15"))     # normal text input from a form
print(safe_chai_bill(2, 15))          # normal numbers
print(safe_chai_bill("two", 15))      # bad input -> caught
print(safe_chai_bill(0, 15))          # too few cups
```

Expected (roughly):
```
45
30
Please enter numbers only.
Cups must be at least 1
```

**Bonus:** wrap the call in lesson 3's Streamlit app so a bad recipe request shows `st.error(...)` instead of crashing.

In [ ]:
# Your solution here
